In [1]:
import numpy as np
import torch as T
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical

eta_origin = 99.34

class PPOMemory:
    # in-built memory for the statemask
    def __init__(self, batch_size):
        self.states = []
        self.probs = []
        self.vals = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.batch_size = batch_size

    def generate_batches(self):
        n_states = len(self.states)
        batch_start = np.arange(0, n_states, self.batch_size)
        indices = np.arange(n_states, dtype=np.int64)
        np.random.shuffle(indices)
        batches = [indices[i:i+self.batch_size] for i in batch_start]

        return np.array(self.states),\
                np.array(self.actions),\
                np.array(self.probs),\
                np.array(self.vals),\
                np.array(self.rewards),\
                np.array(self.dones),\
                batches
    
    def store_memory(self, state, action, probs, vals, reward, done):
        self.states.append(state)
        self.actions.append(action)
        self.probs.append(probs)
        self.vals.append(vals)
        self.rewards.append(reward)
        self.dones.append(done)

    def clear_memory(self):
        self.states = []
        self.probs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.vals = []

In [ ]:
# main masknet implementation for integration from the main repository
import os
eta_origin = 99.34
class ActorNetwork(nn.Module):
    def __init__(self, n_actions, input_dims, alpha,
            fc1_dims=256, fc2_dims=256, chkpt_dir='tmp/ppo'):
        super(ActorNetwork, self).__init__()

        self.checkpoint_file = os.path.join(chkpt_dir, 'actor_torch_ppo')
        self.actor = nn.Sequential(
                nn.Linear(*input_dims, fc1_dims),
                nn.ReLU(),
                nn.Linear(fc1_dims, fc2_dims),
                nn.ReLU(),
                nn.Linear(fc2_dims, n_actions),
                nn.Softmax(dim=-1)
        )

        self.optimizer = optim.Adam(self.parameters(), lr=alpha)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    def forward(self, state):
        dist = self.actor(state)
        dist = Categorical(dist)
        
        return dist

    def save_checkpoint(self):
        T.save(self.state_dict(), self.checkpoint_file)

    def load_checkpoint(self):
        self.load_state_dict(T.load(self.checkpoint_file))

class CriticNetwork(nn.Module):
    def __init__(self, input_dims, beta, fc1_dims=256, fc2_dims=256,
            chkpt_dir='tmp/ppo'):
        super(CriticNetwork, self).__init__()

        self.checkpoint_file = os.path.join(chkpt_dir, 'critic_torch_ppo')
        self.critic = nn.Sequential(
                nn.Linear(*input_dims, fc1_dims),
                nn.ReLU(),
                nn.Linear(fc1_dims, fc2_dims),
                nn.ReLU(),
                nn.Linear(fc2_dims, 1)
        )

        self.optimizer = optim.Adam(self.parameters(), lr=beta)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    def forward(self, state):
        value = self.critic(state)

        return value

    def save_checkpoint(self):
        T.save(self.state_dict(), self.checkpoint_file)

    def load_checkpoint(self):
        self.load_state_dict(T.load(self.checkpoint_file))

class Masknet:
    def __init__(self, n_actions, input_dims, gamma=0.99, alpha=0.0003, beta=0.001, gae_lambda=0.95,
            policy_clip=0.2, batch_size=64, n_epochs=10, chkpt_dir = 'tmp/ppo'):
        self.gamma = gamma
        self.policy_clip = policy_clip
        self.n_epochs = n_epochs
        self.gae_lambda = gae_lambda
        self.LAMBDA = 0
        self.L_RATE_LAMBDA = 1e-3

        self.actor = ActorNetwork(n_actions, input_dims, alpha, chkpt_dir = chkpt_dir)
        self.critic = CriticNetwork(input_dims, beta , chkpt_dir = chkpt_dir)
        self.memory = PPOMemory(batch_size)
    
    def remember(self, state, action, probs, vals, reward, done):
        # store experience into the ppomem buffer 
        self.memory.store_memory(state, action, probs, vals, reward, done)

    def save_models(self):
        print('... saving models ...')
        self.actor.save_checkpoint()
        self.critic.save_checkpoint()

    def load_models(self):
        print('... loading models ...')
        self.actor.load_checkpoint()
        self.critic.load_checkpoint()

    def choose_action(self, observation):
        # output the distribution and the value at the time of predcition of the action
        state = T.tensor([observation], dtype=T.float).to(self.actor.device)
        dist = self.actor(state)
        value = self.critic(state)
        return dist, value
    
    def learn(self, num_mask, disc_score):
        # disc_score is the discounted reward from the original policy
        loss_buff = []
        for _ in range(self.n_epochs):
            # -->pull an experience from the memory
            states, actions, old_probs, vals, rewards, dones, batches = self.memory.generate_batches()
            values = vals
            advantage = np.zeros(len(rewards), dtype=np.float32)
            for t in range(len(rewards)-1):
                # -->calculating advantage and saving
                discount = 1
                adv_t = 0
                for k in range(t, len(rewards)-1):
                    adv_t += discount * (rewards[k] + self.gamma*values[k+1]*(1-int(dones[k]))-values[k])
                    advantage[t] = adv_t
                advantage = T.tensor(advantage).to(self.actor.device)
                values = T.tensor(values).to(self.actor.device)
                for batch in batches:
                    states = T.tensor(states[batch],dtype=np.Float32).to(self.actor.device)
                    old_probs = T.tensor(old_probs[batch]).to(self.actor.device)
                    actions = T.tensor(actions[batch]).to(self.actor.device)
                    dist = self.actor(actions)
                    critic_value = self.critic(actions)
                    critic_value = T.squeeze(critic_value)
                    # --> start building the ppo objective
                    new_probs = dist.log_probs(actions)
                    prob_ratio = new_probs.exp()/old_probs.exp()
                    weighted_probs = advantage[batch] * prob_ratio
                    weighted_clipped_probs = T.clamp(
                        prob_ratio, 1-self.policy_clip,1+self.policy_clip
                    ) * advantage[batch] # does the whole batch have a single advantage? or is it upto the batch which is the t?
                    # --> forming equation 17
                    actor_loss = -T.min(weighted_probs, weighted_clipped_probs).mean() # why mean? because we are doing this over the whole batch?
                    if self.LAMBDA > 1:
                        actor_loss = -actor_loss
                    # formed equation 17 <--
                    # ppo objective for the masknet built <--
                    returns = advantage[batch] + values[batch]
                    critic_loss = ((returns - critic_value)**2).mean()
                    total_loss = actor_loss + 0.5 * critic_loss + 0 * num_mask
                    self.actor.optimizer.zero_grad()
                    self.critic.optimizer.zero_grad()
                    total_loss.backward()
                    self.actor.optimizer.step()
                    self.critic.optimizer.step()
                    loss_buff.append(weighted_probs.cpu().detach().numpy())
                self.LAMBDA -= self.L_RATE_LAMBDA * (2 * eta_origin + np.mean(loss_buff) - 2 * np.mean(disc_score))
                self.LAMBDA = max(self.LAMBDA,0)
            self.memory.clear_memory()